# Customer Segmentation & Behavioral Persona Profiling

This section focuses on the **unsupervised learning** phase of the project. We transform raw transaction logs into distinct customer segments by identifying purchasing patterns, loyalty drivers, and price sensitivities.

### 1. Advanced Feature Engineering
* **Product Insights:** Identification of **Drivers** (high-reach products) and **Niche** items (low reach but high fidelity) to serve as behavioral anchors.
* **Customer Features:** Using `Polars` to compute 10+ high-level metrics per user, including:
    * **Loyalty & Frequency:** `order_frequency`, `reorder_ratio`.
    * **Basket Composition:** `avg_basket_size`, `avg_basket_value`.
    * **Strategic Indices:** **HAI** (Healthy Affinity Index) and **PSI** (Price Sensitivity Index).
    * **Behavioral Affinity:** Dependency on "Driver" products and affinity for "Niche" items.

### 2. Data Preparation & Scaling
* **Outlier Mitigation:** Log-transformation of skewed variables (e.g., basket value) to normalize distributions.
* **Standardization:** Applying `StandardScaler` to ensure all behavioral features contribute equally to the distance metrics.

### 3. Clustering Architecture & Refinement
* **Model Selection:** Implementation of `MiniBatchKMeans` for rapid iteration and standard `KMeans` (k-means++) for the final robust assignment.
* **Optimal K Search:** Utilization of the **Elbow Method** (Inertia) to determine the ideal number of segments ($K=3$).
* **Anomaly Detection:** Statistical identification of **atypical customers** (top 1% by distance from centroids).

### 4. Model Validation & Stability
* **Cohesion:** Calculating the **Silhouette Score** on a representative sample to validate cluster separation.
* **Stability Test:** Measuring the **Adjusted Rand Index (ARI)** between different random seeds to ensure segments are consistent and not artifacts of initialization.
* **Separation:** Heatmap analysis of inter-centroid Euclidean distances.

### 5. Persona Mapping & Strategic Visualization
* **Persona Identification:**
    1.  **The Premium Healths:** Focused on high HAI and quality.
    2.  **The Daily Economizers:** Driven by high PSI and frequency.
    3.  **The Budget-Healthy Mix:** Balanced approach to health and value.
* **Visual Profiling:** * **Radar Charts:** Multi-dimensional "signatures" for each persona.
    * **Market Weight:** Pie chart distribution of the customer base.
    * **Dynamic Profiling:** Automated conversion of raw scores into "Low/Mid/High" business labels.

### 6. Artifact Serialization
* **Export:** Saving the final segmentation results, the trained `StandardScaler`, and the `KMeans` model for production inference.

In [ ]:
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pl.scan_parquet("../data/processed/df_final_for_pipeline.parquet")

# Insight 2: Drivers (Top 10% reach products)
product_reach = df.group_by("product_id").agg(pl.col("user_id").n_unique().alias("reach")).collect()
threshold_driver = product_reach["reach"].quantile(0.9)
driver_ids = product_reach.filter(pl.col("reach") >= threshold_driver)["product_id"].to_list()

# Insight 3: Niche Products (Top 25% fidelity among products with low reach)
product_stats = df.group_by("product_id").agg([
    pl.col("user_id").n_unique().alias("reach"),
    pl.col("reordered").mean().alias("fidelity")
]).collect()
niche_ids = product_stats.filter(
    (pl.col("reach") <= product_stats["reach"].quantile(0.25)) &
    (pl.col("fidelity") >= product_stats["fidelity"].quantile(0.75))
)["product_id"].to_list()

In [ ]:
customer_features = (
    df
    .group_by("user_id")
    .agg(
        total_orders = pl.col("order_id").n_unique(),
        avg_period = pl.col("days_since_prior_order").mean(),
        reorder_ratio = pl.col("reordered").mean(),
        total_spend = pl.col("order_value").sum(),

        avg_basket_size = (
            pl.when(pl.col("order_id").n_unique() > 0)
              .then(pl.col("product_id").count() / pl.col("order_id").n_unique())
              .otherwise(0)
        ),

        weekend_basket_intensity = (
            pl.when(
                pl.col("order_id")
                  .filter(pl.col("order_dow").is_in([0, 1]))
                  .n_unique() > 0
            )
            .then(
                pl.col("department_id")
                  .filter(pl.col("order_dow").is_in([0, 1]))
                  .n_unique()
                /
                pl.col("order_id")
                  .filter(pl.col("order_dow").is_in([0, 1]))
                  .n_unique()
            )
            .otherwise(0)
        ),

        driver_dependency = (
            pl.col("product_id").is_in(driver_ids).sum()
            / pl.col("product_id").count()
        ),

        niche_affinity_score = (
            pl.col("product_id").is_in(niche_ids).mean()
            * pl.col("reordered")
                .filter(pl.col("product_id").is_in(niche_ids))
                .mean()
                .fill_null(0)
        ),

        aisle_penetration = pl.col("department_id").n_unique() / 21.0,
        HAI = pl.col("healthy").mean(),
        PSI = pl.col("cheap_product").mean(),
    )
    .with_columns(
        order_frequency = pl.col("total_orders") / (pl.col("avg_period") + 1),
        avg_basket_value = pl.col("total_spend") / pl.col("total_orders")
    )
    .collect()

)
customer_features = customer_features.with_columns([
    pl.all().fill_null(0)
])

In [ ]:
features_list = [
    'order_frequency', 'reorder_ratio', 'avg_basket_size', 'avg_basket_value',
    'weekend_basket_intensity', 'driver_dependency', 'niche_affinity_score',
    'aisle_penetration', 'HAI', 'PSI'
]

X = customer_features.select(features_list).to_numpy()
print(np.isnan(X).sum())
# Log-transform skewed features (avg_basket_value, avg_basket_size) to reduce the impact of outliers
X[:, 3] = np.log1p(X[:, 3])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

mbk = MiniBatchKMeans(n_clusters=3, batch_size=65536, random_state=42)
customer_features = customer_features.with_columns(
    pl.Series(name="cluster", values=mbk.fit_predict(X_scaled))
)

In [ ]:
inertias = []
for k in range(2, 10):
    km = MiniBatchKMeans(n_clusters=k, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)

plt.plot(range(2, 10), inertias, marker='o')
plt.title("Méthode du Coude")

In [ ]:
# --- ÉTAPE 5 : MODÈLE FINAL ET ATTRIBUTION ---
n_clusters = 3
n_population = X_scaled.shape[0]
n_sample_train = min(1_000_000, n_population)

if n_sample_train < n_population:
    idx_train = np.random.choice(n_population, n_sample_train, replace=False)
    X_train = X_scaled[idx_train]
else:
    X_train = X_scaled

# On utilise UN SEUL objet : km_final
km_final = KMeans(
    n_clusters=n_clusters,
    init='k-means++',
    n_init=20,
    max_iter=500,
    random_state=42,
    tol=1e-4
)

# Entraînement
km_final.fit(X_train)

# Prédiction sur TOUT le dataset
clusters = km_final.predict(X_scaled)
customer_features = customer_features.with_columns(
    pl.Series(name="cluster", values=clusters)
)

# Extraction des centres cohérents
centers_scaled = km_final.cluster_centers_

In [ ]:
dist_to_center = np.sqrt(np.sum((X_scaled - centers_scaled[clusters])**2, axis=1))

threshold = np.percentile(dist_to_center, 99)
customer_features = customer_features.with_columns(
    pl.Series(name="is_atypical", values=(dist_to_center > threshold))
)

In [ ]:
# Never compute silhouette on the whole dataset (13M+ rows) - it's too heavy
idx_sample = np.random.choice(len(X_scaled), 60000, replace=False)
sil_score = silhouette_score(X_scaled[idx_sample], clusters[idx_sample])

print(f"Silhouette score(estimated on 60k pts) : {sil_score:.4f}")
print(f"Number of atypical detected (Top 1%) : {customer_features.filter(pl.col('is_atypical')).shape[0]}")

# --- 6. CLUSTERS SYNTHESE ---
summary = customer_features.group_by("cluster").agg([
    pl.col(f).mean().alias(f"mean_{f}") for f in features_list
]).sort("cluster")

print("\nSummary of clusters :")
print(summary)

# Save results
customer_features.write_parquet("../data/processed/customer_segmentation_results.parquet")

In [ ]:
cluster_names = {
    0: "The Premium Healths",
    1: "The Daily Economizers",
    2: "The Budget-Healthy Mix",
}

summary_df = customer_features.group_by("cluster").agg([
    pl.col(f).mean().alias(f) for f in features_list
]).to_pandas().set_index("cluster")

summary_df['persona'] = summary_df.index.map(cluster_names)

scaler_viz = MinMaxScaler()
summary_scaled = pd.DataFrame(
    scaler_viz.fit_transform(summary_df.drop(columns=['persona'])),
    columns=features_list,
    index=summary_df.index
)
summary_scaled['persona'] = summary_df['persona']

def make_radar_chart_named(df):
    labels = [c for c in df.columns if c != 'persona']
    num_vars = len(labels)

    # Calculating angles
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    cmap = plt.get_cmap('tab10')

    # Using .iterrows() to ensure that the index is linked to the persona
    for i, (idx, row) in enumerate(df.iterrows()):
        values = row.drop('persona').values.flatten().tolist()
        values += values[:1]

        persona_name = row['persona']

        fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
        color = cmap(i % 10)

        ax.fill(angles, values, color=color, alpha=0.25)
        ax.plot(angles, values, color=color, linewidth=2)

        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        ax.set_thetagrids(np.degrees(angles[:-1]), labels)

        # The title now uses the actual ID (idx) and the mapped name (persona_name)
        ax.set_title(f"Persona: {persona_name}\n(Cluster ID: {idx})",
                     size=14, color=color, y=1.1, fontweight='bold')

        plt.show()

temp_data = summary_df.drop(columns=['persona'])
summary_scaled = pd.DataFrame(
    scaler_viz.fit_transform(temp_data),
    columns=temp_data.columns,
    index=summary_df.index
)
summary_scaled['persona'] = summary_df['persona']

make_radar_chart_named(summary_scaled)

HAI (Healthy Affinity Index): Measures the ‘well-being/health’ orientation of the basket.

PSI (Price Sensitivity Index): Measures the ‘economy/lowest price’ orientation of the basket.

In [ ]:
import matplotlib.pyplot as plt

cluster_counts = (
    customer_features.get_column("cluster")
    .value_counts()
    .sort("cluster")
)

ids = cluster_counts["cluster"].to_list()
counts = cluster_counts["count"].to_list()
total = sum(counts)
sizes = [c / total for c in counts]

labels = [f"{cluster_names[i]}" for i in ids]
colors = [plt.cm.tab10(i) for i in ids]

fig, ax = plt.subplots(figsize=(8, 8))

wedges, texts, autotexts = ax.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.80,
    explode=[0.03] * len(labels),
    textprops={'fontweight': 'bold'}
)

plt.setp(autotexts, size=12, color="white")
plt.setp(texts, size=11)

centre_circle = plt.Circle((0,0), 0.60, fc='white')
fig.gca().add_artist(centre_circle)

plt.title("Personas Distribution (Market Weight)", size=16, fontweight='bold', pad=20)
plt.axis('equal')
plt.tight_layout()
plt.show()

print("Volumes réels par cluster :")
print(cluster_counts)

In [ ]:
# 1. Calculation of headcount
counts = customer_features.group_by("cluster").count().to_pandas()
counts['percentage'] = (counts['count'] / counts['count'].sum()) * 100
counts = counts.sort_values("cluster")

counts['name'] = counts['cluster'].map(cluster_names)

print("--- CLUSTER SIZE CHECK ---")
print(counts[['cluster', 'name', 'count', 'percentage']].to_string(index=False))

# 3. Verification of the 5% rule
min_size = counts['percentage'].min()
if min_size < 5:
    print(f"\n⚠️ WARNING: Cluster(s) too small detected ({min_size:.2f}% < 5%)")
    print("Action: Consider reducing K or checking for outliers.")
else:
    print(f"\n✅ SUCCESS: All clusters are > 5% (Min: {min_size:.2f}%)")

In [ ]:
from scipy.spatial.distance import pdist, squareform
import seaborn as sns

# 1. Calculation of Euclidean distances between centres
centers = mbk.cluster_centers_
dist_matrix = squareform(pdist(centers))

# 2. Create a mapping of names for display (optional)
# labels = [cluster_names.get(i, f"Cluster {i}") for i in range(n_clusters)]
labels = [f"C{i}" for i in range(len(centers))]

# 3. Visualisation
plt.figure(figsize=(8, 6))
sns.heatmap(dist_matrix, annot=True, fmt=".2f", cmap="YlGnBu",
            xticklabels=labels, yticklabels=labels)
plt.title("Inter-Centroid Distance (Normalised Space)")
plt.show()

# 4. Diagnostic
min_dist = pdist(centers).min()
avg_dist = pdist(centers).mean()

print(f"Minimum distance between two centres : {min_dist:.2f}")
print(f"Average distance between clusters : {avg_dist:.2f}")

if min_dist < 1.0:
    print("\n⚠️ WARNING: Some clusters are very close. Risk of redundancy.")

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import KMeans
import numpy as np

n_total = X_scaled.shape[0]
n_test = min(500000, n_total)

print(f"Total population for the test : {n_total}")
print(f"Size of sample used : {n_test}")

if n_test < n_total:
    idx_test = np.random.choice(n_total, n_test, replace=False)
    X_test = X_scaled[idx_test]
else:
    X_test = X_scaled

# 3. Run 1 (Seed 42)
km1 = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(X_test)

# 4. Run 2 (Seed 99)
km2 = KMeans(n_clusters=n_clusters, n_init=10, random_state=99).fit(X_test)

# 5. ARI Computation
stability_ari = adjusted_rand_score(km1.labels_, km2.labels_)
print(f"\nNew stability score ARI : {stability_ari:.4f}")

if stability_ari > 0.7:
    print("✅ STABLE RESULT: Your segments are consistent.")
else:
    print("⚠️ UNSTABLE RESULT: Consider reducing K or reviewing your features.")

In [ ]:
import numpy as np

# 1. We start with your summary_df (the one containing the averages per cluster)
# We ensure that the business columns are clear
mapping_colonnes = {
    'order_frequency': 'Loyalty',
    'avg_basket_size': 'Basket',
    'avg_basket_value': 'Price',
    'PSI': 'Nutriscore / Healthy',
    'niche_affinity_score': 'Niche',
    'driver_dependency': 'Driver',
    'weekend_basket_intensity': 'Weekend'
}

# 2. Creating the dynamic table
profiling_dynamique = summary_df.rename(columns=mapping_colonnes)

#3. Function to convert continuous scores into labels (Low/Medium/High)
def get_label(value, col_min, col_max):
    rel_val = (value - col_min) / (col_max - col_min) if col_max > col_min else 0.5
    if rel_val < 0.33: return "Low (+)"
    elif rel_val < 0.66: return "Mid (++)"
    else: return "High (+++)"

# 4. Applying mapping to numeric columns
cols_to_map = list(mapping_colonnes.values())
for col in cols_to_map:
    col_min = profiling_dynamique[col].min()
    col_max = profiling_dynamique[col].max()
    profiling_dynamique[col] = profiling_dynamique[col].apply(lambda x: get_label(x, col_min, col_max))

profiling_dynamique['persona'] = profiling_dynamique.index.map(cluster_names)

cols = ['persona'] + cols_to_map
profiling_final = profiling_dynamique[cols].reset_index()

print("Dynamic Profiling Table (Automated)")
display(profiling_final)

In [ ]:
# 1. Calculation of centres in the original space
centers_original = scaler.inverse_transform(mbk.cluster_centers_)

# 2. Creating a clean DataFrame
df_centers = pd.DataFrame(
    centers_original,
    columns=features_list,
    index=[f"Cluster {i}" for i in range(len(centers_original))]
)

# 3. Added the names of the Personas for clarity.
df_centers['persona'] = df_centers.index.map(lambda x: cluster_names[int(x.split()[-1])])

# 4. Display cleaning (rounded for business reading)
df_centers_display = df_centers.copy()
# Prices are rounded to two decimal places and frequencies to one decimal place.
for col in df_centers_display.columns:
    if col != 'persona':
        df_centers_display[col] = df_centers_display[col].map(lambda x: round(x, 2))

print("📊 Actual characteristics per cluster (unscaled data)")
display(df_centers_display)

In [ ]:
import os
import pandas as pd

df_personas_for_goldenbasket = pd.DataFrame(
    centers_original,
    columns=features_list
)

# 2. Add the identification columns
# .index retrieves the IDs (0, 1, 2)
df_personas_for_goldenbasket['cluster_id'] = df_personas_for_goldenbasket.index
df_personas_for_goldenbasket['persona'] = df_personas_for_goldenbasket['cluster_id'].map(cluster_names)

#3. Reorganisation of columns (ID and Persona first)
cols = ['cluster_id', 'persona'] + [c for c in features_list if c not in ['cluster_id', 'persona']]
df_personas_for_goldenbasket = df_personas_for_goldenbasket[cols]

output_path = "../data/processed/df_personas_for_goldenbasket.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_personas_for_goldenbasket.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Success! The file has been dynamically generated from the template: {output_path}")
display(df_personas_for_goldenbasket.head())

In [ ]:
import joblib

model_dir = "../models/"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(scaler, os.path.join(model_dir, "scaler_customers.joblib"))

joblib.dump(km_final, os.path.join(model_dir, "kmeans_model_final.joblib"))